# SEMI: experience classifier (HCIN revision)

Run on Google Colab with a GPU (Runtime > Change runtime type > T4 GPU). Upload `gold_standard_582.csv` and `corpus_for_inference.csv` when prompted.

Outputs: `cv_metrics.json`, `cv_predictions.csv`, `corpus_predictions.csv`. Send these three files back.

Settings follow the manuscript: bert-base-uncased, 4 epochs, batch size 16, learning rate 2e-5. Evaluation uses stratified 5-fold cross-validation on the 582 human-adjudicated reviews; the final model is trained on all 582 and applied to the corpus.

In [ ]:
!pip -q install transformers==4.44.2 accelerate scikit-learn
import transformers, torch, sklearn; print(transformers.__version__, torch.__version__, torch.cuda.is_available())

In [ ]:
from google.colab import files
up = files.upload()  # select gold_standard_582.csv and corpus_for_inference.csv

In [ ]:
import numpy as np, pandas as pd, torch, random, json, os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, cohen_kappa_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
SEED=42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
MODEL='bert-base-uncased'; EPOCHS=4; BS=16; LR=2e-5; MAXLEN=128
dev=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gold=pd.read_csv('gold_standard_582.csv'); gold['text']=gold.text.astype(str)
tok=AutoTokenizer.from_pretrained(MODEL)
print(gold.label.value_counts().sort_index())

In [ ]:
def encode(texts):
    return tok(list(texts), truncation=True, padding=True, max_length=MAXLEN, return_tensors='pt')
def train(texts, labels, seed):
    set_seed(seed)
    model=AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=6).to(dev)
    enc=encode(texts); y=torch.tensor(list(labels))
    n=len(y); opt=torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps=EPOCHS*((n+BS-1)//BS); sch=get_linear_schedule_with_warmup(opt, int(0.1*steps), steps)
    model.train()
    for ep in range(EPOCHS):
        perm=torch.randperm(n)
        for i in range(0,n,BS):
            idx=perm[i:i+BS]
            out=model(input_ids=enc['input_ids'][idx].to(dev), attention_mask=enc['attention_mask'][idx].to(dev), labels=y[idx].to(dev))
            out.loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); sch.step(); opt.zero_grad()
    return model
@torch.no_grad()
def predict(model, texts, bs=128):
    model.eval(); P=[]
    texts=list(texts)
    for i in range(0,len(texts),bs):
        e=tok(texts[i:i+bs], truncation=True, padding=True, max_length=MAXLEN, return_tensors='pt').to(dev)
        P.append(torch.softmax(model(**e).logits,-1).cpu().numpy())
    return np.vstack(P)

In [ ]:
skf=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof=np.zeros(len(gold),dtype=int); fold_rows=[]
for k,(tr,te) in enumerate(skf.split(gold.text, gold.label)):
    m=train(gold.text.iloc[tr], gold.label.iloc[tr], SEED+k)
    pr=predict(m, gold.text.iloc[te]).argmax(1); oof[te]=pr
    fold_rows.append({'fold':k+1,'accuracy':accuracy_score(gold.label.iloc[te],pr),'macro_f1':f1_score(gold.label.iloc[te],pr,average='macro'),'weighted_f1':f1_score(gold.label.iloc[te],pr,average='weighted')})
    print(fold_rows[-1]); del m; torch.cuda.empty_cache()
folds=pd.DataFrame(fold_rows); print(folds.describe().loc[['mean','std']])
rep=classification_report(gold.label, oof, digits=3, output_dict=True)
print(classification_report(gold.label, oof, digits=3))
cm=confusion_matrix(gold.label, oof, labels=range(6)); print(cm)
json.dump({'folds':fold_rows,'mean':folds.mean(numeric_only=True).to_dict(),'std':folds.std(numeric_only=True).to_dict(),
           'pooled_report':rep,'pooled_kappa_vs_gold':cohen_kappa_score(gold.label,oof),'confusion_matrix':cm.tolist(),
           'settings':{'model':MODEL,'epochs':EPOCHS,'batch':BS,'lr':LR,'maxlen':MAXLEN,'seed':SEED,'transformers':transformers.__version__}},
          open('cv_metrics.json','w'), indent=2)
gold.assign(pred=oof).to_csv('cv_predictions.csv', index=False)

In [ ]:
final=train(gold.text, gold.label, SEED)
corpus=pd.read_csv('corpus_for_inference.csv'); corpus['text']=corpus.text.astype(str)
P=predict(final, corpus.text)
corpus['pred']=P.argmax(1); corpus['prob']=P.max(1)
corpus[['doc_id','dest','pred','prob']].to_csv('corpus_predictions.csv', index=False)
print(corpus.pred.value_counts().sort_index())

In [ ]:
files.download('cv_metrics.json'); files.download('cv_predictions.csv'); files.download('corpus_predictions.csv')